# NLP Fase 3 — Comparativa TF-IDF vs LSTM vs BERT

## Dataset

In [4]:
import pandas as pd
texts = [
    'me encanta este curso',
    'odio este producto',
    'es un curso normal',
    'no me gusta nada'
]
labels2 = ['POS', 'NEG', 'NEU', 'NEG']
label_map = {'NEG': 0, 'NEU': 1, 'POS': 2}
labels = [label_map[label] for label in labels2]
df = pd.DataFrame({'text': texts, 'label2': labels2, 'label': labels})
df


,text,label2,label
0,me encanta este curso,POS,2
1,odio este producto,NEG,0
2,es un curso normal,NEU,1
3,no me gusta nada,NEG,0


## 1. TF-IDF baseline

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = TfidfVectorizer()
X = tfidf.fit_transform(df['text'])
model_tfidf = LogisticRegression(max_iter=1000)
model_tfidf.fit(X, df['label'])
preds_tfidf = model_tfidf.predict(X)
inv_label_map = {value: key for key, value in label_map.items()}
df['tfidf_label'] = [inv_label_map[pred] for pred in preds_tfidf]
df[['text', 'label2', 'tfidf_label']]


,text,label2,tfidf_label
0,me encanta este curso,POS,NEG
1,odio este producto,NEG,NEG
2,es un curso normal,NEU,NEU
3,no me gusta nada,NEG,NEG


## 2. LSTM

In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['text'])
seq = tokenizer.texts_to_sequences(df['text'])
padded = pad_sequences(seq)

model_lstm = tf.keras.Sequential([
    tf.keras.layers.Embedding(1000, 16),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(3, activation='softmax')
])

model_lstm.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_lstm.fit(padded, df['label'], epochs=10, verbose=0)
preds_lstm = model_lstm.predict(padded).argmax(axis=1)
df['lstm_label'] = [inv_label_map[int(pred)] for pred in preds_lstm]
df[['text', 'label2', 'lstm_label']]


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 216ms/step


,text,label2,lstm_label
0,me encanta este curso,POS,NEG
1,odio este producto,NEG,NEG
2,es un curso normal,NEU,NEG
3,no me gusta nada,NEG,NEG


## 3. BERT (Transformers)

In [7]:
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model='pysentimiento/robertuito-sentiment-analysis',
    tokenizer='pysentimiento/robertuito-sentiment-analysis',
    framework='pt'
)
results = classifier(df['text'].tolist(), truncation=True)
df['bert_label'] = [item['label'] for item in results]
df[['text', 'label2', 'bert_label']]


model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
df[['text', 'label2', 'tfidf_label', 'lstm_label', 'bert_label']]


In [ ]:
# TODO: comparar resultados
# - accuracy
# - errores
# - diferencias

## Autoevaluación

In [ ]:
# TODO:
# 1. ¿Qué modelo funciona mejor?
# 2. ¿Por qué?
# 3. ¿Cuál usarías en producción?